In [ ]:
# ==============================================================================
# Reprojecting LAS (Point Cloud) Tiles to BNG/ODN
# Transformation: OSTN15 (Horizontal) & OSGM15 (Vertical)
# From: EPSG:32630 (WGS 1984 UTM 30N)
# To:   EPSG:27700 (BNG) + ODN Height
# Date: April 2026
# ==============================================================================

In [ ]:
%pip install pyproj laspy -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
## Test if the OSTN15 and OSGM15 grids are working
import os
import pyproj
from pyproj import Transformer

  ## Setup Paths
  # Local Users: Leave it as BASE_DIR = ""
  # Colab Users: Use your Drive path below (i.e BASE_DIR = "/content/drive/MyDrive/Folder_Name/")

  BASE_DIR = "/content/drive/MyDrive/Folder_Name/"

# Point PyProj to your local/Drive grids folder
grid_folder = os.path.join(BASE_DIR, "Transformation_Grids")
pyproj.datadir.append_data_dir(grid_folder)

# Anchor the grid paths directly
osgm15_path = os.path.abspath(os.path.join(grid_folder, "uk_os_osgm15_gb.tif"))
ostn15_path = os.path.abspath(os.path.join(grid_folder, "ostn15_etrs_to_osgb.gsb"))

# Build the pipeline
t = Transformer.from_pipeline(
    "+proj=pipeline "
    "+step +inv +proj=utm +zone=30 +ellps=WGS84 "
    f"+step +inv +proj=vgridshift +grids={osgm15_path} +multiplier=1 "
    f"+step +proj=hgridshift +grids={ostn15_path} "
    "+step +proj=tmerc +lat_0=49 +lon_0=-2 +k=0.9996012717 "
    "+x_0=400000 +y_0=-100000 +ellps=airy +units=m +no_defs"
)

# Transform a test point (UTM Easting, UTM Northing, Ellipsoidal Height)
e_bng, n_bng, z_odn = t.transform(456000.0, 5850000.0, 75.0)

print("Validation Completed!!!")
print(f"BNG:   {e_bng:.3f}, {n_bng:.3f}")
print(f"Z ODN: {z_odn:.3f}")

In [ ]:
import os
import glob
import laspy
import numpy as np
import pyproj
from pyproj import Transformer, CRS

# 1. DEFINE THE FUNCTION
def reproject_point_cloud_bng_odn(input_folder, output_folder, grid_folder):
    """
    Reprojects .las files from UTM 30N (WGS84) to BNG (OSGB36) + ODN (Newlyn).
    Compatible with both laspy v1.x and v2.x.
    """
    # Tell PyProj where to find custom grids
    pyproj.datadir.append_data_dir(grid_folder)

    # Anchor the grid paths directly
    osgm15_path = os.path.abspath(os.path.join(grid_folder, "uk_os_osgm15_gb.tif"))
    ostn15_path = os.path.abspath(os.path.join(grid_folder, "ostn15_etrs_to_osgb.gsb"))

    # Set the Pipeline: Forward horizontal shift, Inverse vertical shift
    z_shift_pipeline = (
        "+proj=pipeline "
        "+step +inv +proj=utm +zone=30 +ellps=WGS84 "
        f"+step +inv +proj=vgridshift +grids={osgm15_path} +multiplier=1 "
        f"+step +proj=hgridshift +grids={ostn15_path} "
        "+step +proj=tmerc +lat_0=49 +lon_0=-2 +k=0.9996012717 "
        "+x_0=400000 +y_0=-100000 +ellps=airy +units=m +no_defs"
    )
    transformer = Transformer.from_pipeline(z_shift_pipeline)

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    las_files = sorted(glob.glob(os.path.join(input_folder, "*.las")))
    total_files = len(las_files)
    print(f"Found {total_files} files. Starting transformation...")

    for index, las_path in enumerate(las_files, 1):
        fname = os.path.basename(las_path)
        try:
            las = laspy.read(las_path)

            if len(las.points) > 0:
                # Transform the coordinates
                x_out, y_out, z_out = transformer.transform(np.array(las.x), np.array(las.y), np.array(las.z))

                valid_mask = np.isfinite(x_out) & np.isfinite(y_out) & np.isfinite(z_out)

                if np.any(valid_mask):
                    # Create the new LAS object
                    new_las = laspy.create(point_format=las.header.point_format, file_version=las.header.version)

                    try:
                    # Attempt to add CRS for laspy v2.0+
                        new_las.header.add_crs(CRS.from_epsg(27700))
                    except Exception:
                        pass

                    # Scale and Offset
                    new_las.header.offsets = [np.floor(np.min(x_out[valid_mask])),
                                              np.floor(np.min(y_out[valid_mask])),
                                              np.floor(np.min(z_out[valid_mask]))]
                    new_las.header.scales = [0.001, 0.001, 0.001]

                    # Assign the transformed coordinates
                    new_las.x = x_out[valid_mask]
                    new_las.y = y_out[valid_mask]
                    new_las.z = z_out[valid_mask]

                    # Transfer extra attributes (Colors, Intensity, etc.)
                    for dim in las.point_format.dimension_names:
                        if dim not in ['X', 'Y', 'Z', 'x', 'y', 'z']:
                            new_las[dim] = las[dim][valid_mask]

                    # Save the file
                    out_name = fname.replace(".las", "_BNG_ODN.las")
                    new_las.write(os.path.join(output_folder, out_name))

            if index % 50 == 0 or index == total_files:
                print(f"Progress: {index}/{total_files} tiles completed...")

        except Exception as e:
            print(f"❌ Error in {fname}: {e}")

    print(f"\n--- SUCCESS: Reprojection Complete ---")

# 2. EXECUTE THE SCRIPT
if __name__ == "__main__":

    ## Setup Paths
    # Local Users: Leave it as BASE_DIR = ""
    # Colab Users: Use your Drive path below (i.e BASE_DIR = "/content/drive/MyDrive/Folder_Name/")

    BASE_DIR = "/content/drive/MyDrive/Folder_Name/"

    input_dir = os.path.join(BASE_DIR, 'Point_Cloud') # Replace "Point_Cloud" with your input las folder name
    output_dir = os.path.join(BASE_DIR, 'Point_Cloud_BNG_ODN') # Replace "Point_Cloud_BNG_ODN'" with your preferred output folder name
    grid_dir = os.path.join(BASE_DIR, 'Transformation_Grids') # Replace "Transformation_Grids" with the name of the folder you kept the grid files

    # Run the function
    reproject_point_cloud_bng_odn(input_dir, output_dir, grid_dir)

In [ ]:
# Check if there are tiles containing infinity values
# If none, then the reprojection was successful!
import os
import glob
import laspy
import numpy as np

## Setup Paths
# Local Users: Leave it as BASE_DIR = ""
# Colab Users: Use your Drive path below (i.e BASE_DIR = "/content/drive/MyDrive/Folder_Name/")

BASE_DIR = "/content/drive/MyDrive/Folder_Name/"

search_path = os.path.join(BASE_DIR, "Point_Cloud_BNG_ODN", "*.las")  # Replace "Point_Cloud_BNG_ODN'" with the name you set in the previous cell
bad_files = []
for las_path in glob.glob(search_path):
    las = laspy.read(las_path)
    if not np.all(np.isfinite(las.z)):
        bad_files.append(las_path)

print(f"{len(bad_files)} files with inf values:", bad_files)